# Experiment 1 — Does gauge, or dimension, set the recoverability boundary?

**Purpose.** The paper's Spearman analysis cannot separate gauge freedom from irrep
dimension, because in the regular representation gauge is a monotone function of
dimension (numerically: gauge = block rank in *every* regular-rep block; verified in
Step 0 below). This notebook (i) re-analyzes recoverability at the level of isotypic
*blocks* instead of groups, and (ii) runs the decoupling experiment: the same irrep at
multiplicities m = 1, 2, 3, 4, so gauge is swept 1 → 4 → 9 → 16 at *fixed* dimension.

---
## PRE-REGISTERED ANALYSIS PLAN — frozen before running. Do not edit after results exist.

**Unit of analysis.** The real isotypic block (Step 1) / the planted invariant copy (Step 2).

**Outcomes.**
- `sigma_star_det`: per-seed first noise level at which the block's structure fails the
  spectrum-preserving rotation null at alpha = 0.01 (B = 199 rotations); block summary =
  median over seeds. Right-censored at the grid maximum.
- `delta`: invariance defect (Eq. 3 of the paper) over group generators.
- Step 2 only — `gauge_drift` = block_containment − copy_fidelity: energy that stayed in
  the isotypic block but left the planted copy, i.e. structure *rotated within its
  commutant*. This is the direct operationalization of the mechanism.

**Primary hypotheses.**
- H1 (Step 1): partial Spearman rho(sigma_star, gauge | irrep_dim) < 0 across blocks.
  NOTE: in the regular rep gauge ≡ block rank, so H1 is only informative through the
  handful of FS-type wedges (complex pairs, Q8); Step 2 is the decisive test.
- H2 (Step 2): at fixed sigma and fixed irrep dimension d = 3 (S4 standard), gauge_drift
  increases monotonically in multiplicity m (Page trend test, one-sided), and is
  identically ~0 at m = 1 (no commutant to rotate within).
- H2b (secondary): the invariance defect delta does NOT increase with m — commutant
  rotations preserve invariance, so delta is predicted to be blind to the gauge
  mechanism. If delta *does* rise with m, the mechanism story needs revision.

**Inference.** Step 1: cluster bootstrap over groups (2000 resamples, 95% CI) for the
partial Spearman. Step 2: Page's L across m (seeds as subjects) + pairwise permutation
tests between adjacent m (10,000 permutations).

**Decision rule (pre-committed).**
- gauge_drift rises with m at fixed d (H2 holds) → 'gauge governs' is earned, with the
  mechanism = rotation within the commutant; the paper's framing sharpens.
- flat in m → gauge and dimension are empirically inseparable → keep 'organizes'.
- Whichever cell this lands in is the paper. No post-hoc framing selection.


In [ ]:
# ---------------- configuration ----------------
SMOKE = True          # True: fast sanity pass (~minutes). False: full run (hours).

import numpy as np

if SMOKE:
    N_SEEDS      = 8      # noise seeds per block / condition
    B_ROT        = 99     # rotations in the detection null (p floor 0.01 -> use alpha 0.05 in smoke)
    ALPHA        = 0.05
    SIGMA_GRID   = np.linspace(0.0, 2.0, 11)
    STEP2_SEEDS  = 40
    STEP2_SIGMAS = [0.15, 0.3, 0.5]
    STEP2_MULTS  = [1, 2, 3, 4]
    N_BOOT       = 500
else:
    N_SEEDS      = 50
    B_ROT        = 199
    ALPHA        = 0.01
    SIGMA_GRID   = np.linspace(0.0, 2.0, 21)
    STEP2_SEEDS  = 200
    STEP2_SIGMAS = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
    STEP2_MULTS  = [1, 2, 3, 4, 5, 6]
    N_BOOT       = 2000

MASTER_SEED = 20260706
rng = np.random.default_rng(MASTER_SEED)
print("SMOKE mode:", SMOKE)

In [ ]:
"""Core machinery for Experiment 1: groups, regular representations,
numeric character tables (Dixon-style), isotypic projectors, commutant dims.

Everything is verified numerically (group axioms, projector completeness,
orthogonality relations) so errors surface immediately rather than as wrong
science downstream.
"""
import itertools
import numpy as np

# ---------------------------------------------------------------------------
# Group construction: each group is a list of hashable elements + mult function
# ---------------------------------------------------------------------------

def cyclic(n):
    elems = list(range(n))
    mult = lambda a, b: (a + b) % n
    gens = [1]
    return elems, mult, gens, f"Z/{n}"

def dihedral(n):
    # elements (k, f): rotation by k, flip flag f. r=(1,0), s=(0,1)
    elems = [(k, f) for f in (0, 1) for k in range(n)]
    def mult(a, b):
        k1, f1 = a; k2, f2 = b
        if f1 == 0:
            return ((k1 + k2) % n, f2)
        else:
            return ((k1 - k2) % n, 1 - f2)
    gens = [(1, 0), (0, 1)]
    return elems, mult, gens, f"D{n}"

def symmetric(n):
    elems = [tuple(p) for p in itertools.permutations(range(n))]
    def mult(a, b):  # (a*b)(x) = a(b(x))
        return tuple(a[b[i]] for i in range(n))
    gens = [tuple([1, 0] + list(range(2, n)))]
    if n > 2:
        gens.append(tuple(list(range(1, n)) + [0]))  # n-cycle
    return elems, mult, gens, f"S{n}"

def _parity(p):
    p = list(p); par = 0; seen = [False]*len(p)
    for i in range(len(p)):
        if seen[i]: continue
        j, clen = i, 0
        while not seen[j]:
            seen[j] = True; j = p[j]; clen += 1
        par ^= (clen - 1) & 1
    return par

def alternating(n):
    elems = [tuple(p) for p in itertools.permutations(range(n)) if _parity(p) == 0]
    def mult(a, b):
        return tuple(a[b[i]] for i in range(n))
    # generators: 3-cycles (0 1 2) and (0 1 2 ... ) pattern; use (012) and (0,1)(2,..n-1) style
    g1 = tuple([1, 2, 0] + list(range(3, n)))
    if n % 2 == 1:
        g2 = tuple(list(range(1, n)) + [0])           # n-cycle (even perm for odd n)
    else:
        g2 = tuple([0] + list(range(2, n)) + [1])     # (n-1)-cycle on 1..n-1
    return elems, mult, [g1, g2], f"A{n}"

def quaternion8():
    # elements (s, u): s in {+1,-1}, u in {'1','i','j','k'}
    units = ['1', 'i', 'j', 'k']
    tab = {  # unit multiplication: (u,v) -> (sign, unit)
        ('1','1'):(1,'1'),('1','i'):(1,'i'),('1','j'):(1,'j'),('1','k'):(1,'k'),
        ('i','1'):(1,'i'),('i','i'):(-1,'1'),('i','j'):(1,'k'),('i','k'):(-1,'j'),
        ('j','1'):(1,'j'),('j','i'):(-1,'k'),('j','j'):(-1,'1'),('j','k'):(1,'i'),
        ('k','1'):(1,'k'),('k','i'):(1,'j'),('k','j'):(-1,'i'),('k','k'):(-1,'1'),
    }
    elems = [(s, u) for u in units for s in (1, -1)]
    def mult(a, b):
        s1, u1 = a; s2, u2 = b
        s3, u3 = tab[(u1, u2)]
        return (s1 * s2 * s3, u3)
    gens = [(1, 'i'), (1, 'j')]
    return elems, mult, gens, "Q8"


class Group:
    """Finite group with Cayley table, regular rep, classes, characters."""
    def __init__(self, elems, mult, gens, name):
        self.name = name
        self.elems = list(elems)
        self.n = len(self.elems)
        self.idx = {e: i for i, e in enumerate(self.elems)}
        self._verify_and_build(mult, gens)

    def _verify_and_build(self, mult, gens):
        n, idx, elems = self.n, self.idx, self.elems
        # Cayley table
        C = np.empty((n, n), dtype=np.int32)
        for i, a in enumerate(elems):
            for j, b in enumerate(elems):
                C[i, j] = idx[mult(a, b)]
        self.cayley = C
        # identity
        eid = [i for i in range(n) if all(C[i, j] == j for j in range(n))]
        assert len(eid) == 1, f"{self.name}: identity not unique"
        self.e = eid[0]
        # inverses + closure implicit in table construction; associativity spot check
        rng = np.random.default_rng(0)
        for _ in range(200):
            a, b, c = rng.integers(0, n, 3)
            assert C[C[a, b], c] == C[a, C[b, c]], f"{self.name}: not associative"
        inv = np.empty(n, dtype=np.int32)
        for i in range(n):
            j = np.where(C[i] == self.e)[0]
            assert len(j) == 1
            inv[i] = j[0]
        self.inv = inv
        # generators generate the whole group
        gset = {self.e}
        frontier = [self.e]
        gidx = [idx[g] for g in gens]
        while frontier:
            new = []
            for x in frontier:
                for g in gidx:
                    y = C[g, x]
                    if y not in gset:
                        gset.add(y); new.append(y)
            frontier = new
        assert len(gset) == n, f"{self.name}: generators don't generate group"
        self.gens = gidx
        # conjugacy classes
        assigned = np.full(n, -1, dtype=np.int32)
        classes = []
        for i in range(n):
            if assigned[i] >= 0: continue
            orb = set()
            for x in range(n):
                orb.add(C[C[x, i], inv[x]])
            k = len(classes)
            for j in orb: assigned[j] = k
            classes.append(sorted(orb))
        self.classes = classes
        self.class_of = assigned
        self.r = len(classes)

    # -------------------- regular representation --------------------
    def regular_rep(self, g):
        """L(g): permutation matrix, L(g) e_h = e_{gh}."""
        n = self.n
        M = np.zeros((n, n))
        M[self.cayley[g, np.arange(n)], np.arange(n)] = 1.0
        return M

    # -------------------- numeric character table (Dixon) --------------------
    def character_table(self, seed=1, tol=1e-8):
        """Returns chars: (r x r) complex array, chars[t, i] = chi_t(class i),
        verified against orthogonality relations."""
        n, r, classes = self.n, self.r, self.classes
        reps = [c[0] for c in classes]
        sizes = np.array([len(c) for c in classes])
        # structure constants a[i][j][k]: #{(x,y) in C_i x C_j : xy = rep_k}
        A = np.zeros((r, r, r))
        Cay = self.cayley
        for i in range(r):
            for x in classes[i]:
                row = Cay[x]            # x*y for all y
                for j in range(r):
                    prods = row[classes[j]]
                    for k in range(r):
                        A[i, j, k] += np.count_nonzero(prods == reps[k])
        # eigenvector method: w (per character) satisfies M_i w = w_i w, M_i = A[i]
        rng = np.random.default_rng(seed)
        for attempt in range(20):
            c = rng.standard_normal(r)
            M = np.tensordot(c, A, axes=(0, 0))
            evals, evecs = np.linalg.eig(M)
            if np.min(np.abs(evals[:, None] - evals[None, :]) + np.eye(r)) > 1e-6:
                break
        chars = np.zeros((r, r), dtype=complex)
        for t in range(r):
            v = evecs[:, t]
            # identity class index
            id_cls = self.class_of[self.e]
            v = v / v[id_cls]                      # w_identity = 1
            denom = np.sum(np.abs(v) ** 2 / sizes)
            d = np.sqrt(n / denom)
            d = np.round(d.real) if abs(d - np.round(d.real)) < 1e-6 else d
            chars[t] = d * v / sizes
        # sort by degree then verify orthogonality
        order = np.argsort([c[self.class_of[self.e]].real for c in chars])
        chars = chars[order]
        G_inner = (chars * sizes) @ chars.conj().T / n
        assert np.allclose(G_inner, np.eye(r), atol=1e-7), \
            f"{self.name}: character orthogonality failed"
        degs = chars[:, self.class_of[self.e]].real
        assert abs(np.sum(degs ** 2) - n) < 1e-6, f"{self.name}: sum d^2 != |G|"
        self.chars = chars
        self.degrees = np.round(degs).astype(int)
        self.class_sizes = sizes
        return chars

    def fs_indicators(self):
        """Frobenius-Schur indicator per complex irrep: +1 real, 0 complex, -1 quaternionic."""
        sq_class = np.array([self.class_of[self.cayley[g, g]] for g in range(self.n)])
        fs = np.array([np.sum(self.chars[t][sq_class]).real / self.n
                       for t in range(self.r)])
        fs = np.round(fs).astype(int)
        assert np.all(np.isin(fs, [-1, 0, 1])), f"{self.name}: bad FS indicators"
        self.fs = fs
        return fs

    # -------------------- real isotypic blocks of the regular rep -------------
    def real_isotypic_blocks(self):
        """Group complex irreps into real blocks (complex-conjugate pairs merged).
        Returns list of dicts with projector, dims, multiplicity, gauge (numeric)."""
        if not hasattr(self, 'chars'): self.character_table()
        if not hasattr(self, 'fs'): self.fs_indicators()
        n, r = self.n, self.r
        Ls = [self.regular_rep(g) for g in range(n)]
        char_by_elem = self.chars[:, self.class_of]          # r x n
        used = np.zeros(r, dtype=bool)
        blocks = []
        for t in range(r):
            if used[t]: continue
            partners = [t]
            if self.fs[t] == 0:
                # find conjugate partner
                for s in range(r):
                    if s != t and not used[s] and \
                       np.allclose(self.chars[s], self.chars[t].conj(), atol=1e-7):
                        partners.append(s); break
                assert len(partners) == 2, f"{self.name}: conj partner missing"
            for s in partners: used[s] = True
            P = np.zeros((n, n))
            for s in partners:
                d = self.degrees[s]
                coef = (d / n) * char_by_elem[s].conj()
                P += np.real(sum(coef[g] * Ls[g] for g in range(n)))
            # verify projector
            assert np.allclose(P @ P, P, atol=1e-9), f"{self.name}: block not idempotent"
            rank = int(np.round(np.trace(P)))
            d_complex = int(self.degrees[t])
            fs = int(self.fs[t])
            blocks.append(dict(group=self.name, irrep_dim=d_complex, fs=fs,
                               partners=partners, P=P, block_rank=rank))
        # completeness
        Ptot = sum(b['P'] for b in blocks)
        assert np.allclose(Ptot, np.eye(n), atol=1e-8), f"{self.name}: blocks incomplete"
        self.blocks = blocks
        return blocks

    def commutant_dim(self, P, tol=1e-7):
        """Numeric real commutant dimension of the regular action restricted to
        range(P): dim{ M : M rho(g) = rho(g) M for generators }."""
        rank = int(np.round(np.trace(P)))
        # orthonormal basis of range(P)
        U, s, _ = np.linalg.svd(P)
        B = U[:, :rank]
        rows = []
        for g in self.gens:
            R = B.T @ self.regular_rep(g) @ B          # rank x rank
            # constraint (R^T kron I - I kron R) vec(M) = 0
            K = np.kron(R.T, np.eye(rank)) - np.kron(np.eye(rank), R)
            rows.append(K)
        K = np.vstack(rows)
        sv = np.linalg.svd(K, compute_uv=False)
        return int(np.sum(sv < tol * max(1.0, sv[0]))), B

    def multiplicity_in_regular(self, t):
        return int(self.degrees[t])   # regular rep: m = d (complex)


def all_eleven_groups():
    specs = [cyclic(6), cyclic(12), dihedral(3), dihedral(4), quaternion8(),
             dihedral(5), dihedral(6), alternating(4), symmetric(4),
             alternating(5), symmetric(5)]
    # note: S3 ~= D3; use D3 construction labelled S3 to match the paper's list
    out = []
    for elems, mult, gens, name in specs:
        if name == "D3": name = "S3"
        out.append(Group(elems, mult, gens, name))
    return out


In [ ]:
"""Experiment engine for Exp 1 (gauge vs dimension).

Outcome definitions (frozen; see pre-registration cell in notebook):

- Planted weights: W0 = orthonormal basis of the target subspace (n x r),
  noise model W_sigma = W0 + sigma * E, E_ij ~ N(0, 1/sqrt(n)) so that the
  expected column norm of sigma*E is sigma (noise scale comparable across n).

- Detection: isotypic energy fraction f_P(W) = ||P W||_F^2 / ||W||_F^2 against
  the spectrum-preserving rotation null f_P(Q W), Q ~ Haar(O(n)).
  p = (1 + #{f_P(QW) >= f_P(W)}) / (B + 1).  Detected iff p <= alpha.

- sigma*_det (per seed): first sigma on the grid where detection fails,
  after isotonic cleanup (a single spurious re-detection at higher sigma does
  not resurrect); summary = median over seeds.

- Invariance defect delta(W): per the paper's Eq. 3, computed on the
  orthonormalized column space, averaged over GENERATORS (not all g) for
  speed -- verified to be a monotone proxy for the all-g version.

- Copy-fidelity decomposition (Step 2): planted copy U (one d-dim invariant
  copy inside a multiplicity-m isotypic block):
    block_containment = ||P_block Uhat||_F^2 / d      (stayed in the block)
    copy_fidelity     = ||U^T Uhat||_F^2 / d          (stayed the SAME copy)
    gauge_drift       = block_containment - copy_fidelity
  gauge_drift is energy that remained in the isotypic block but moved to a
  different (gauge-equivalent) copy: "rotated within its commutant".
"""
import numpy as np


def haar_orthogonal(n, rng):
    A = rng.standard_normal((n, n))
    Q, R = np.linalg.qr(A)
    return Q * np.sign(np.diag(R))


def energy_fraction(P, W):
    num = np.linalg.norm(P @ W) ** 2
    den = np.linalg.norm(W) ** 2
    return num / den


def rotation_null_pvalue(P, W, B, rng):
    f_obs = energy_fraction(P, W)
    count = 0
    n = W.shape[0]
    for _ in range(B):
        Q = haar_orthogonal(n, rng)
        if energy_fraction(P, Q @ W) >= f_obs:
            count += 1
    return (1 + count) / (B + 1)


def invariance_defect(W, rep_mats):
    """delta over the supplied representation matrices (use generators)."""
    Q, _ = np.linalg.qr(W)
    Pr = Q @ Q.T
    nrm = np.linalg.norm(Pr)
    vals = [np.linalg.norm(L @ Pr - Pr @ L) / nrm for L in rep_mats]
    return float(np.mean(vals))


def orthonormal_basis(P):
    r = int(np.round(np.trace(P)))
    U, s, _ = np.linalg.svd(P)
    return U[:, :r]


def sigma_star_detection(P, W0, sigma_grid, alpha, B, n_seeds, rng):
    """Per-seed sigma* then median. Returns (median_sigma_star, per_seed array,
    detection_rate_per_sigma)."""
    n = W0.shape[0]
    det = np.zeros((n_seeds, len(sigma_grid)), dtype=bool)
    for s in range(n_seeds):
        E = rng.standard_normal(W0.shape) / np.sqrt(n)
        for j, sig in enumerate(sigma_grid):
            W = W0 + sig * E
            p = rotation_null_pvalue(P, W, B, rng)
            det[s, j] = p <= alpha
    # isotonic cleanup: once failed, stays failed
    stars = np.empty(n_seeds)
    for s in range(n_seeds):
        ok = det[s].copy()
        fail = np.where(~ok)[0]
        if len(fail) == 0:
            stars[s] = sigma_grid[-1]          # right-censored
        else:
            stars[s] = sigma_grid[fail[0]]     # first failure
    return float(np.median(stars)), stars, det.mean(axis=0)


# ---------------- Step 2: multiplicity construction for S_n standard irrep ----
def standard_irrep_matrices(group, n_points):
    """S_n acting on n_points via its natural permutation action, restricted
    to the sum-zero subspace -> the (n_points-1)-dim standard irrep, as
    explicit orthogonal matrices for every group element."""
    # group.elems are permutation tuples of range(n_points)
    ones = np.ones(n_points) / np.sqrt(n_points)
    # orthonormal basis of sum-zero subspace
    Bz = np.linalg.svd(np.eye(n_points) - np.outer(ones, ones))[0][:, :n_points - 1]
    mats = []
    for p in group.elems:
        Pm = np.zeros((n_points, n_points))
        for i in range(n_points):
            Pm[p[i], i] = 1.0
        mats.append(Bz.T @ Pm @ Bz)
    return mats  # list of (n_points-1) x (n_points-1) orthogonal matrices


def multiplicity_rep(std_mats, m, ambient, rng):
    """Representation on R^ambient containing the standard irrep with
    multiplicity exactly m: rho = std^{oplus m} oplus trivial^{oplus rest},
    conjugated by a fixed random orthogonal basis so nothing is axis-aligned.
    Returns (rep matrices per element, block projector P_std, one-copy basis U)."""
    d = std_mats[0].shape[0]
    assert m * d <= ambient
    Qamb = haar_orthogonal(ambient, rng)
    reps, P = [], np.zeros((ambient, ambient))
    for M in std_mats:
        R = np.eye(ambient)
        for c in range(m):
            sl = slice(c * d, (c + 1) * d)
            R[sl, sl] = M
        reps.append(Qamb @ R @ Qamb.T)
    P[:m * d, :m * d] = np.eye(m * d)
    P = Qamb @ P @ Qamb.T
    U = Qamb[:, :d]                       # one specific copy
    return reps, P, U


def copy_decomposition(U, W, P_block):
    """Given noisy W spanning ~the planted copy U (n x d), decompose where the
    energy went."""
    Q, _ = np.linalg.qr(W)
    d = U.shape[1]
    block = np.linalg.norm(P_block @ Q) ** 2 / d
    copy = np.linalg.norm(U.T @ Q) ** 2 / d
    return dict(block_containment=block, copy_fidelity=copy,
                gauge_drift=block - copy)


# ---------------- statistics -------------------------------------------------
def partial_spearman(y, x, z):
    """Spearman partial correlation of y with x controlling z (all 1-d)."""
    from scipy.stats import rankdata
    ry, rx, rz = rankdata(y), rankdata(x), rankdata(z)
    def resid(a, b):
        b1 = np.column_stack([np.ones_like(b), b])
        coef, *_ = np.linalg.lstsq(b1, a, rcond=None)
        return a - b1 @ coef
    ey, ex = resid(ry, rz), resid(rx, rz)
    denom = (np.linalg.norm(ey) * np.linalg.norm(ex))
    if denom == 0:
        return 0.0
    return float(ey @ ex / denom)


def cluster_bootstrap_ci(df_rows, stat_fn, cluster_key, n_boot=2000, seed=0,
                         alpha=0.05):
    """Nonparametric cluster bootstrap: resample clusters with replacement."""
    rng = np.random.default_rng(seed)
    clusters = sorted({r[cluster_key] for r in df_rows})
    by_c = {c: [r for r in df_rows if r[cluster_key] == c] for c in clusters}
    stats = []
    for _ in range(n_boot):
        pick = rng.choice(clusters, size=len(clusters), replace=True)
        sample = [r for c in pick for r in by_c[c]]
        try:
            stats.append(stat_fn(sample))
        except Exception:
            continue
    stats = np.array(stats)
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(np.mean(stats)), (float(lo), float(hi))


def page_trend_L(matrix):
    """Page's L statistic for ordered alternatives. matrix: subjects x
    conditions, conditions in hypothesized increasing order. Returns (L, z)."""
    from scipy.stats import rankdata
    n, k = matrix.shape
    ranks = np.vstack([rankdata(row) for row in matrix])
    R = ranks.sum(axis=0)
    L = float(np.sum(np.arange(1, k + 1) * R))
    EL = n * k * (k + 1) ** 2 / 4
    VL = n * k ** 2 * (k + 1) * (k ** 2 - 1) / 144
    z = (L - EL) / np.sqrt(VL)
    return L, float(z)


## Step 0 — Per-block inventory + the structural-confound check

Builds all eleven groups, verifies the machinery (orthogonality, projector
completeness — assertions inside will halt on failure), and tabulates every real
isotypic block with its irrep dimension, FS type, block rank, **numerically computed**
gauge (commutant dimension), and real multiplicity.

The key thing to look at: in the regular representation, is gauge distinguishable from
block rank *at all*? (Expected answer: no — they coincide in every block, which is the
structural confound the paper's Spearman inherits. The wedge is multiplicity: Q8's block
has the same gauge as D4's but multiplicity 1 vs 2.)

In [ ]:
groups = all_eleven_groups()
block_rows = []
for G in groups:
    G.character_table(); G.fs_indicators()
    for b in G.real_isotypic_blocks():
        gauge, _ = G.commutant_dim(b['P'])
        d, fs = b['irrep_dim'], b['fs']
        mult = d if fs in (0, 1) else d // 2     # real multiplicity in regular rep
        block_rows.append(dict(group=G.name, n=G.n, irrep_dim=d, fs=fs,
                               block_rank=b['block_rank'], gauge=gauge,
                               multiplicity=mult, P=b['P'], G=G))

import pandas as pd
tab = pd.DataFrame([{k: v for k, v in r.items() if k not in ('P', 'G')}
                    for r in block_rows])
print(tab.to_string(index=False))
print("\ngauge == block_rank in all blocks:",
      bool((tab.gauge == tab.block_rank).all()))
print("blocks where multiplicity separates from gauge (the wedge):")
print(tab[(tab.gauge == 4)][['group','irrep_dim','fs','gauge','multiplicity']].to_string(index=False))

## Step 1 — Per-block sigma* under the calibrated detector

For every block: plant W = orthonormal basis of the block, add Gaussian noise across
the sigma grid, run the spectrum-preserving rotation null, extract per-seed sigma*,
plus the invariance defect at two reference noise levels. Then the discriminating
statistic: partial Spearman of sigma* against gauge controlling for irrep dimension,
with group-level cluster bootstrap.

**Runtime warning:** the FULL setting is hours (S5 blocks are 120x120 with 199
rotations per grid point per seed). SMOKE gives the shape of the answer in minutes.

In [ ]:
results = []
for r in block_rows:
    G, P = r['G'], r['P']
    if r['irrep_dim'] == 1 and r['fs'] == 1 and r['block_rank'] == 1 and r['group'] not in ('Z/6','Z/12'):
        pass   # keep everything; trivial blocks are cheap and are honest data points
    W0 = orthonormal_basis(P)
    grid = SIGMA_GRID.copy()
    med, stars, rate = sigma_star_detection(P, W0, grid, ALPHA, B_ROT,
                                            N_SEEDS, rng)
    while med >= grid[-1] and grid[-1] < 16.0:   # right-censored: extend grid
        grid = np.linspace(0.0, grid[-1] * 2, len(SIGMA_GRID))
        med, stars, rate = sigma_star_detection(P, W0, grid, ALPHA, B_ROT,
                                                N_SEEDS, rng)
    gen_mats = [G.regular_rep(g) for g in G.gens]
    deltas = {}
    for sig in (0.1, 0.2):
        vals = []
        for s in range(N_SEEDS):
            E = rng.standard_normal(W0.shape) / np.sqrt(G.n)
            vals.append(invariance_defect(W0 + sig * E, gen_mats))
        deltas[sig] = float(np.mean(vals))
    results.append(dict(group=r['group'], irrep_dim=r['irrep_dim'], fs=r['fs'],
                        block_rank=r['block_rank'], gauge=r['gauge'],
                        multiplicity=r['multiplicity'], sigma_star=med,
                        sigma_star_all=stars.tolist(),
                        delta01=deltas[0.1], delta02=deltas[0.2]))
    print(f"{r['group']:5s} d={r['irrep_dim']} fs={r['fs']:+d} rank={r['block_rank']:2d} "
          f"gauge={r['gauge']:2d} m={r['multiplicity']}  sigma*={med:.2f}  "
          f"delta(0.2)={deltas[0.2]:.3f}")

res = pd.DataFrame([{k: v for k, v in d.items() if k != 'sigma_star_all'}
                    for d in results])
res.to_csv('exp1_step1_blocks.csv', index=False)

In [ ]:
# Discriminating statistics for Step 1 (interpret with the Step-0 caveat in mind)
sub = res.copy()

rho_gauge_given_d = partial_spearman(sub.sigma_star.values,
                                     sub.gauge.values, sub.irrep_dim.values)
rho_mult_given_d  = partial_spearman(sub.sigma_star.values,
                                     sub.multiplicity.values, sub.irrep_dim.values)
print(f"partial Spearman rho(sigma*, gauge | d)        = {rho_gauge_given_d:+.3f}")
print(f"partial Spearman rho(sigma*, multiplicity | d) = {rho_mult_given_d:+.3f}")

rows = sub.to_dict('records')
def stat_gauge(sample):
    import numpy as np
    y = np.array([r['sigma_star'] for r in sample])
    x = np.array([r['gauge'] for r in sample])
    z = np.array([r['irrep_dim'] for r in sample])
    return partial_spearman(y, x, z)
mean_g, ci_g = cluster_bootstrap_ci(rows, stat_gauge, 'group', n_boot=N_BOOT,
                                    seed=MASTER_SEED)
print(f"cluster-bootstrap (over groups): rho(sigma*, gauge | d) = {mean_g:+.3f}, "
      f"95% CI ({ci_g[0]:+.3f}, {ci_g[1]:+.3f})")
print("\nREAD WITH CARE: within the regular rep gauge == block rank, so this partial")
print("correlation is powered almost entirely by the FS wedge cases. Step 2 is decisive.")

## Step 2 — The decoupling experiment (decisive)

S4's 3-dimensional standard irrep, constructed at multiplicity m = 1 … M inside a fixed
24-dimensional ambient space (`standard^{⊕m} ⊕ trivial^{⊕(24−3m)}`, conjugated by a
random orthogonal basis so nothing is axis-aligned). Gauge of the standard block is
exactly m² (asserted numerically). We plant **one copy** (a specific 3-dim invariant
subspace), add noise, and measure where the energy goes.

Predictions (pre-registered):
- `gauge_drift` ≡ 0 at m = 1 (mathematically forced: no other copy exists);
- `gauge_drift` strictly increasing in m at every fixed sigma if gauge governs;
- `delta` flat in m (commutant rotations preserve invariance).

In [ ]:
elems, mult_fn, gens, name = symmetric(4)
S4 = Group(elems, mult_fn, gens, name)
std = standard_irrep_matrices(S4, 4)
AMBIENT = 24

step2 = []
for m in STEP2_MULTS:
    if 3 * m > AMBIENT: break
    reps, P_blk, U = multiplicity_rep(std, m, AMBIENT, rng)
    # assert constructed gauge = m^2
    B = orthonormal_basis(P_blk)
    K = np.vstack([np.kron((B.T @ reps[g] @ B).T, np.eye(B.shape[1]))
                   - np.kron(np.eye(B.shape[1]), (B.T @ reps[g] @ B))
                   for g in S4.gens])
    sv = np.linalg.svd(K, compute_uv=False)
    gauge = int(np.sum(sv < 1e-7 * sv[0]))
    assert gauge == m * m, f"constructed gauge {gauge} != m^2"
    gen_mats = [reps[g] for g in S4.gens]
    for sig in STEP2_SIGMAS:
        for s in range(STEP2_SEEDS):
            E = rng.standard_normal(U.shape) / np.sqrt(AMBIENT)
            W = U + sig * E
            out = copy_decomposition(U, W, P_blk)
            out.update(m=m, gauge=gauge, sigma=sig, seed=s,
                       delta=invariance_defect(W, gen_mats))
            step2.append(out)

s2 = pd.DataFrame(step2)
summary = s2.groupby(['sigma', 'm'])[['gauge_drift', 'delta',
                                      'block_containment', 'copy_fidelity']].mean()
print(summary.round(4).to_string())
s2.to_csv('exp1_step2_multiplicity.csv', index=False)

In [ ]:
# Step 2 inference: Page trend across m per sigma + adjacent-m permutation tests
from itertools import combinations
rng_perm = np.random.default_rng(MASTER_SEED + 1)

for sig in STEP2_SIGMAS:
    d_sig = s2[s2.sigma == sig]
    mats = d_sig.pivot_table(index='seed', columns='m', values='gauge_drift').values
    L, zL = page_trend_L(mats)
    line = f"sigma={sig}: Page z (gauge_drift increasing in m) = {zL:+.2f}"
    # adjacent-m permutation tests
    ms = sorted(d_sig.m.unique())
    for a, b in zip(ms[:-1], ms[1:]):
        xa = d_sig[d_sig.m == a].gauge_drift.values
        xb = d_sig[d_sig.m == b].gauge_drift.values
        obs = xb.mean() - xa.mean()
        pool = np.concatenate([xa, xb]); nA = len(xa)
        perm = np.array([(lambda p: p[nA:].mean() - p[:nA].mean())
                         (rng_perm.permutation(pool)) for _ in range(2000)])
        p = (1 + np.sum(perm >= obs)) / 2001
        line += f"  m{a}->m{b}: diff={obs:+.4f} p={p:.4f}"
    print(line)

print("\nH2b check (delta should be flat in m):")
print(s2.groupby(['sigma','m']).delta.mean().round(4).to_string())

In [ ]:
# plots
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for sig in STEP2_SIGMAS:
    g = s2[s2.sigma == sig].groupby('m').gauge_drift.agg(['mean', 'sem'])
    axes[0].errorbar(g.index, g['mean'], yerr=g['sem'], marker='o', label=f'sigma={sig}')
    d = s2[s2.sigma == sig].groupby('m').delta.mean()
    axes[1].plot(d.index, d.values, marker='s', label=f'sigma={sig}')
axes[0].set_xlabel('multiplicity m (gauge = m^2, irrep dim fixed = 3)')
axes[0].set_ylabel('gauge drift'); axes[0].set_title('rotation within the commutant')
axes[1].set_xlabel('multiplicity m'); axes[1].set_ylabel('invariance defect delta')
axes[1].set_title('delta is blind to the gauge mechanism (prediction: flat)')
for a in axes: a.legend()
plt.tight_layout(); plt.savefig('exp1_step2.png', dpi=150)
print('saved exp1_step2.png')

## Interpretation (pre-committed — read the plan cell before this one)

- **H2 holds** (gauge_drift ↑ in m, ~0 at m=1, Page z large at every sigma): gauge
  governs recoverability *in the identifiability sense* — noise rotates structure among
  gauge-equivalent embeddings, and the amount of rotation is set by the commutant. This
  is the 'governs' evidence, and the mechanism paragraph writes itself.
- **H2b holds alongside** (delta flat in m): important nuance for the paper — the
  *invariance defect used in the current draft cannot see this mechanism*. The paper's
  Figure-1 ordering is a dimension effect; the gauge effect lives in copy identifiability.
  Both are real; they are different claims and the text must separate them.
- **H2 fails** (flat drift): gauge and dimension are inseparable even under direct
  manipulation → the abstract keeps 'organizes' and gains a paragraph saying this
  comparison was run.

Optional follow-ups (do not block the paper): S5 standard irrep replication
(swap `symmetric(4)`→`symmetric(5)`, 5 points, d=4); learned-weight version (train a
small net on S4 acting on 4 points) if grokking cooperates.